## Alpha Earth Extraction

In [1]:
import pandas as pd
import geopandas as gpd
import numpy as np
import matplotlib.pyplot as plt
import time
import datetime
import ee
import geemap.foliumap as geemap  
import datetime

In [ ]:
# Authenticate ee. You first need to set up a free Google Cloud account
ee.Authenticate()

In [ ]:
# Initialise Earth Engine
ee.Initialize()  

In [ ]:
# plot problem location

i=10617                     
i=10642               
i=10625

lat = gdf.iloc[i,1]
lon = gdf.iloc[i,2]
end_year = str(gdf.iloc[i,4])
start_date = end_year + '-01-01'
end_date = end_year + '-12-31'

print(lon,lat,end_year,start_date,end_date)

point = ee.Geometry.Point(lon, lat)    
box = ee.Geometry.BBox((lon-0.001), (lat-0.001), (lon+0.001), (lat+0.001))             
box_fc =  ee.FeatureCollection(box).style(**{'color':'blue','fillColor': '#00000000'}) 

image = (
    ee.ImageCollection('GOOGLE/SATELLITE_EMBEDDING/V1/ANNUAL')     
    .filterDate(start_date, end_date)           
    .filterBounds(box)
)

sample = image.median().sample(point, scale=10)
sample_dict = sample.first().toDictionary().getInfo()
sample_np = np.array(list(sample_dict.values()), dtype=float)

Map = geemap.Map(height='400', width='40%')
Map.centerObject(point,zoom=14)

vis = {'min':-0.3, 'max':0.3, 'bands': ['A01','A02','A03']}     
Map.addLayer(image,vis,'image') 
Map.addLayer(box_fc, {}, "box_fc")

# Add centre point
Map.addLayer(point,{"color": "red"}, "point")    

Map

## Read GLANCE geodataframe-points for samples

In [ ]:
# read geodataframe from file
gdf_biomeA_extracted = gpd.read_parquet('../Glance_data/biomeA_extracted.parquet') 
#gdf_biomeB_extracted = gpd.read_parquet('../Glance_data/biomeB_extracted.parquet') 
gdf_biomeC_extracted = gpd.read_parquet('../Glance_data/biomeC_extracted.parquet') 
gdf_biomeD_extracted = gpd.read_parquet('../Glance_data/biomeD_extracted.parquet') 
gdf_biomeE_extracted = gpd.read_parquet('../Glance_data/biomeE_extracted.parquet') 
#gdf_biomeF_extracted = gpd.read_parquet('../Glance_data/biomeF_extracted.parquet') 

In [ ]:
######### update this for each run ##############
gdf = gdf_biomeA_extracted.copy()   
len(gdf)

In [ ]:
gdf.head(2)

In [ ]:
# How many GLANCE samples per year
print('Samples per year')
sum=0
for i in range(2017,2025):
    y=np.sum((gdf['End_Year'] == i))
    print(i, '=', y)
    sum=sum+y
print('\ntotal samples =',sum)

In [ ]:
# How many samples per landcover class

level3 = ['Water(0)','Developed(1)','Barren(2)','Trees(3)','Shrub(4)','Grassland(5)','Agriculture(6)'] 

sum=0
for i in range(0,7):
    y=np.sum((gdf['level3'] == i))
    print(i, level3[i], '=', y)
    sum=sum+y
print('Total Samples =',sum,'\n')

In [ ]:
# remove 2017 samples
gdf_remove = gdf[gdf["End_Year"] != 2017]      # drop samples from 2017
print(len(gdf_remove))

In [ ]:
###########################################################################
gdf_new = gdf.copy()
len(gdf),len(gdf_new)

In [ ]:
# OLD CELL - Extract Pixel Training Data from Copernicus S2 Collection
print(datetime.datetime.now())

time_start = time.time()

print('no of training samples =',len(gdf))

start_sample = 10000
#end_sample = 10000
end_sample = len(gdf)

print('from',start_sample,'to',end_sample)

sent_all = np.zeros((0,4))
missing_samples=[]

for i in range(start_sample,end_sample):                               
    sample_no = gdf.iloc[i,0]
    lat = gdf.iloc[i,1]
    lon = gdf.iloc[i,2]
    end_year = str(gdf.iloc[i,4])
    start_date = end_year + '-01-01'
    end_date = end_year + '-12-31'

    point = ee.Geometry.Point(lon, lat)   

    collection = (
        ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')       # whole Sentinel-2 collection
        .filterDate(start_date, end_date)                       # time period - end year of GLANCE sample
        .filterBounds(point)                                    # only images interesecting this feature
    )

    num_images = collection.size().getInfo()     # no of images will be the same for 10m and 20m bands

    if (num_images == 0):
        print(i, ' sample:',sample_no, ' year:',end_year, ' images = 0. no image will be added. remove gdf row')
        gdf_new = gdf_new.drop(gdf.index[i])
        print('gdf_new length:',len(gdf_new))
        missing_samples.append([sample_no,end_year])                   
    else:

#        bands = ['B2','B3','B4','B8','B5','B6','B7','B8A','B11','B12']
        bands = ['B2','B3','B4','B8']
        coll_bands = collection.select(bands)                               # All bands of interest, 10m 
        
        # calculate the median of the bands. 
        median_bands =  coll_bands.median()                                  # Median is single multi-band image
                                                                        
        sample = median_bands.select(bands).sample(point, scale=10).first()
        sample_dict = sample.toDictionary().getInfo()
        sent_vector = np.array(list(sample_dict.values()), dtype=float)
    
        # order is B11,B12,B2,B3,B4,B5,B6,B7,B8,B8A
        #order = [2, 3, 4, 5, 6, 7, 8, 9, 0, 1]     # new order of indices
        #sent_vector = sent_vector[order]
    
        sent_vector = np.expand_dims(sent_vector,0)  

        sent_all = np.concatenate((sent_all,sent_vector),axis=0)
    
        if i % 100 == 0: 
            print(i, ' sample:',sample_no, ' year:',end_year, ' sent_vector:', sent_vector.shape)

print('\n\nsent_all: ',sent_all.shape)

time_end = time.time()
print('\n\nfinal run time=',(time_end-time_start)/60," minutes")


## Extract Training Data from AlphaEarth Foundation Model

In [ ]:
print(datetime.datetime.now())
time_start = time.time()

print('no of training samples =',len(gdf))

start_sample = 0
end_sample = len(gdf)
end_sample = 10

print('from',start_sample,'to',end_sample)

sample_all = np.zeros((0,64))

for i in range(start_sample,end_sample):                               
    sample_no = gdf.iloc[i,0]
    lat = gdf.iloc[i,1]
    lon = gdf.iloc[i,2]
    end_year = str(gdf.iloc[i,4])
    start_date = end_year + '-01-01'
    end_date = end_year + '-12-31'

    point = ee.Geometry.Point(lon, lat)   


    point = ee.Geometry.Point(lon, lat)    
    box = ee.Geometry.BBox((lon-0.001), (lat-0.001), (lon+0.001), (lat+0.001))             
    box_fc =  ee.FeatureCollection(box).style(**{'color':'blue','fillColor': '#00000000'}) 

    image = (
        ee.ImageCollection('GOOGLE/SATELLITE_EMBEDDING/V1/ANNUAL')     
        .filterDate(start_date, end_date)           
        .filterBounds(box)                       # using box rather than points helps if point is directly on image boundary
    )

    sample = image.median().sample(point, scale=10)
    sample_dict = sample.first().toDictionary().getInfo()
    sample_np = np.array(list(sample_dict.values()), dtype=float)

    sample_np = np.expand_dims(sample_np,0)  
    sample_all = np.concatenate((sample_all,sample_np),axis=0)

    if i % 100 == 0: 
        print(i, ' sample=',sample_no)

print('\nsample_all: ',sample_all.shape)

###############################################################################

# save pixel data
np.save('../Pixel_data/alpha_pixels_BiomeA.npy', sample_all)

# save labels
labels = gdf["level3"].to_numpy().astype(int)
print('labels: ', labels.shape)
np.save('../Pixel_data/alpha_pixels_BiomeA_labels.npy', labels)

print('data saved')

time_end = time.time()
print('\n\nfinal run time=',(time_end-time_start)/60," minutes")


In [ ]:
sample_all.shape

## Save/Load Pixel Data to File

In [2]:
model_name = 'alpha_pixels_BiomeA'
#model_name = 'sent_pixels_biome13'

file_name='../Pixel_data/' + model_name + '.npy'
file_name

'../Pixel_data/alpha_pixels_BiomeA.npy'

In [3]:
# load pixel data
pixel_loaded = np.load(file_name)
pixel_loaded.shape

(20834, 64)

In [ ]:
sample_concat = np.concatenate((pixel_loaded,sample_all),axis=0)
sample_concat.shape

In [ ]:
# save pixel data
np.save(file_name, sample_concat)

## Save Labels to File

In [ ]:
labels = gdf["level3"].to_numpy().astype(int)

In [ ]:
labels.min(), labels.max()

In [4]:
file_name='../Pixel_data/' + model_name + '_labels.npy'
file_name

'../Pixel_data/alpha_pixels_BiomeA_labels.npy'

In [ ]:
np.save(file_name, labels)

In [5]:
# load labels
labels_loaded = np.load(file_name)
labels_loaded.shape

(20834,)